## Get GPT-4O Response

#### Convert pdf to image

In [65]:
import openai
import json
import os
import time
import base64
from pdf2image import convert_from_bytes
from io import BytesIO
from PIL import Image
from PIL import ImageEnhance

In [66]:
def enhance_image(image):
    img = image.convert("L")  # Grayscale
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(2.0)  # Increase contrast
    return img

def pdf_to_base64_image(pdf_bytes):
    """Convert all pages of a PDF to a list of base64-encoded PNG images."""
    images = convert_from_bytes(pdf_bytes, dpi=300)
    base64_images = []
    for image in images:
        enhanced = enhance_image(image)
        buffer = BytesIO()
        enhanced.save(buffer, format="PNG")
        img_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
        base64_images.append(f"data:image/png;base64,{img_str}")
    return base64_images

#### Invoice Schema

In [67]:
invoice_schema = {
    "name": "extract_invoice_data",
    "description": "Extract structured invoice data from an unstructured invoice text.",
    "parameters": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string","description": "Unique identifier for the invoice"},
            "invoice_date": {"type": "string", "description": "Date when the invoice was issued"},
            "invoice_series": {"type": "string", "description": "Series identifier for the invoice"},
            "supply_date": {"type": "string", "description": "Date when the goods or services were supplied"},
            "provider": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "address": {"type": "string"},
                    "contact": {
                        "type": "object",
                        "properties": {
                            "email": {"type": ["string", "null"]},
                            "phone": {"type": ["string", "null"]},
                            "fax": {"type": ["string", "null"]},
                            "mobile": {"type": ["string", "null"]}
                        },
                        "required": ["email", "phone", "fax", "mobile"]
                    },
                    "VAT_NUMBER/NIE/CIF": {"type": "string","description": "VAT number or similar tax ID such as NIE or CIF"}
                },
                "required": ["name", "address", "contact", "VAT_NUMBER/NIE/CIF"],
                 "description": "Information about the service or goods provider"
            },
            "payment_method": {
                "type": "string",
                "description": "Method used to pay the invoice, such as 'transferencia bancaria', 'efectivo', 'tarjeta de crédito', or 'PayPal'"
            },
            "IBAN": {
                "type": "string",
                "description": "International Bank Account Number where the payment should be sent"
            },
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                       "code":{"type":"string"},
                        "quantity": {"type": "number"},
                        "description": {"type": "string"},
                         "english_description": {
                            "type": "string",
                            "description": "English translation of the item description"
                        },
                        "isServiceOrProduct": {
                            "type": "string",
                            "enum": ["product", "service"],
                            "description": "Classify the item based on its description as either a physical product or a service"
                        },            
                        "category": {
                            "type": "string",
                            "description": "Suggested category for the item based on the description"
                        },
                        "aiSuggestedCategory": {
                            "type": "string",
                            "description": "A detailed and specific category suggested by the AI based solely on the item's description. Not limited to your predefined list."
                        },
                        "matchedCategory": {
                            "type": "string",
                            "description": "Select the closest match from the predefined category list based on the item description",
                            "enum": [
                                "Networking Equipment",
                                "Radio & Communication",
                                "Common Use",
                                "Tools",
                                "Fasteners & Anchoring Hardware",
                                "Cables",
                                "Computers & Accessories",
                                "Telecom Equipment",
                                "Connectors & Adapters",
                                "Phones & Communication Devices",
                                "Fiber Optic Equipment",
                                "Power Supply & Management",
                                "Tapes & Insulation",
                                "Cameras & Surveillance"
                            ]

                        },
                        "genericCategory": {
                            "type": "string",
                            "description": "A broad or high-level category inferred from the item type, useful for reporting or classification at a general level."
                        },
                        "unit": {"type": ["string", "null"]},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"},
                    },
                    "required": ["code","quantity", "description", "unit_price", "total", "project","isServiceOrProduct","english_description","category","aiSuggestedCategory","matchedCategory","genericCategory"]
                }
            },
            "total": {
                "type": "object",
                "properties": {
                    "base_amount": {"type": "number"},
                    "VAT_rate": {"type": "number"},
                    "VAT_amount": {"type": "number"},
                    "IRPF_rate": {"type": ["number", "null"]},
                    "IRPF_amount": {"type": ["number", "null"]},
                    "total_invoice": {"type": "number"}
                },
                "required": ["base_amount", "VAT_rate", "VAT_amount", "IRPF_rate", "IRPF_amount", "total_invoice"]
            },
            "description": {"type": "string"},
            "ocr_understanding_confidence": {
                "type": "integer",
                "minimum": 0,
                "maximum": 100,
                "description": "An estimate (0-100) of how confident the model is in having accurately extracted all required information from the OCR image(s)."
            }
        },
        "required": ["invoice_number", "invoice_date", "provider", "items", "total", "description","ocr_understanding_confidence"]
    }
}

#### Extract data

In [68]:
def extract_invoice_data_from_pdf(pdf_bytes: bytes) -> dict | None:
    openai.api_key = os.getenv("OPENAI_API_KEY")

    try:
        image_base64_list = pdf_to_base64_image(pdf_bytes)
        t1 = time.perf_counter()

        messages = []

        messages.append({
            "role": "system",
            "content": "You are a professional Spanish invoice analyzer. Your task is to extract structured invoice data from OCR-scanned invoices, using a predefined schema."
        })

        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                    """
                        Analyze the following invoice image(s) and extract all relevant data using the function schema provided.

                            📌 Rules:
                            - The **client** is always fixed: DEL-INTERNET TELECOM, S.L.U., NIF B55606446. Do NOT extract client info.
                            - Only extract data related to the **provider**, **invoice**, and **items**.

                            For each item:
                            • Classify it as a "product" or "service" (field: `isServiceOrProduct`)
                            • Translate its description to English (`english_description`)
                            • Suggest a specific category based on the item (field: `aiSuggestedCategory`) — can be freeform
                            • Match it to one of the approved categories listed below (field: `matchedCategory`) — choose the closest relevant match
                            • Suggest a broad/general category (field: `genericCategory`) based on the item's purpose

                            📌 Also include:
                            • An integer `ocr_understanding_confidence` (0–100) estimating your confidence in extraction accuracy.

                            Respond **only** with data that fits the function schema.
                        """
                    )
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_base64_list[0] 
                    }
                }
            ]
        })

        for img_str in image_base64_list[1:]:
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": img_str
                        }
                    }
                ]
            })

        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            functions=[invoice_schema],
            function_call={"name": "extract_invoice_data"},
            temperature=0
        )

        t2 = time.perf_counter()
        print(f"⏱️ OpenAI API call time: {t2 - t1:.2f} seconds")

        function_response = response.choices[0].message.function_call

        if function_response and function_response.arguments:
            parsed_args = json.loads(function_response.arguments)
            return parsed_args
        else:
            print("❌ GPT-4o could not parse JSON, returning raw output")
            return None

    except Exception as e:
        print("❌ GPT-4o Vision extraction failed:", e)
        return None

## Database Connection

In [69]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("supplier_invoices.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS invoices (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Source_File TEXT,
        Invoice_Number TEXT,
        Invoice_Date TEXT,
        Invoice_Series TEXT,       
        Invoice_Supply_Date TEXT,      
        Provider_Name TEXT,
        Provider_Email TEXT,
        Provider_Phone TEXT,
        Provider_Fax TEXT,
        Provider_Mobile TEXT,
        Provider_Address TEXT,
        Provider_VAT TEXT,
        Item_Code TEXT,
        Item_Description TEXT,
        Item_English_Description TEXT,
        Item_Category TEXT,
        Item_AiSuggested_Category TEXT,       
        Item_Matched_Category TEXT,      
        Item_Generic_Category TEXT,     
        Item_Type TEXT CHECK(Item_Type IN ('product', 'service')),
        Item_Quantity REAL,
        Item_Unit TEXT,
        Item_Unit_Price REAL,
        Discount_Rate REAL,
        Discount_Amount REAL,
        Item_VAT_Rate REAL,
        Item_VAT_Amount REAL,
        Item_IRPF_Rate REAL,
        Item_IRPF_Amount REAL,
        Item_Total REAL,
        Base_Amount REAL,
        Total_VAT_Rate REAL,
        Total_VAT_Amount REAL,
        Total_IRPF_Rate REAL,
        Total_IRPF_Amount REAL,
        Total_Invoice REAL,
        Payment_Method TEXT,       
        IBAN TEXT,
        Invoice_Description TEXT,
        OCR_Confidence TEXT,
        Error Text       
    )
""")

conn.commit()
conn.close()


In [70]:
def write_invoices_to_db_structured(invoice_data_list,error):
    # Flatten data into rows (same as you did for CSV)
    
    if isinstance(invoice_data_list, str):
        try:
            invoice_data_list = json.loads(invoice_data_list)
        except json.JSONDecodeError as e:
            print("❌ JSON decoding failed:", e)
            return

    if not isinstance(invoice_data_list, list):
        invoice_data_list = [invoice_data_list] 

    rows = []

    for invoice in invoice_data_list:
        provider = invoice["provider"]
        provider_id = provider["VAT_NUMBER/NIE/CIF"]

        for item in invoice.get("items", []):
           
            row = {
                "Source_File": invoice.get("source_file", ""),
                "Invoice_Number": invoice.get("invoice_number"),
                "Invoice_Date": invoice.get("invoice_date"),
                "Invoice_Series": invoice.get("Invoice_Series"),       
                "Invoice_Supply_Date": invoice.get("Invoice_Supply_Date"), 
                "Provider_Name": provider["name"],
                "Provider_Email": provider.get("contact", {}).get("email"),
                "Provider_Phone": provider.get("contact", {}).get("phone"),
                "Provider_Fax": provider.get("contact", {}).get("fax"),
                "Provider_Mobile": provider.get("contact", {}).get("mobile"),
                "Provider_Address": provider["address"],
                "Provider_VAT": provider_id,
                "Item_Code":item.get("code"),
                "Item_Description": item.get("description"),
                "Item_English_Description": item.get("english_description"),
                "Item_Category": item.get("category"),
                "Item_AiSuggested_Category": item.get("aiSuggestedCategory"),       
                "Item_Matched_Category": item.get("matchedCategory"),       
                "Item_Generic_Category": item.get("genericCategory"),      
                "Item_Type": item.get("isServiceOrProduct"),
                "Item_Quantity": item.get("quantity"),
                "Item_Unit": item.get("unit"),
                "Item_Unit_Price": item.get("unit_price"),
                "Discount_Rate": item.get("discount_rate"),
                "Discount_Amount": item.get("discount_amount"),
                "Item_VAT_Rate": item.get("VAT_rate"),
                "Item_VAT_Amount": item.get("VAT_amount"),
                "Item_IRPF_Rate": item.get("IRPF_rate"),
                "Item_IRPF_Amount": item.get("IRPF_amount"),
                "Item_Total": item.get("total"),
                "Base_Amount": invoice["total"].get("base_amount"),
                "Total_VAT_Rate": invoice["total"].get("VAT_rate"),
                "Total_VAT_Amount": invoice["total"].get("VAT_amount"),
                "Total_IRPF_Rate": invoice["total"].get("IRPF_rate"),
                "Total_IRPF_Amount": invoice["total"].get("IRPF_amount"),
                "Total_Invoice": invoice["total"].get("total_invoice"),
                "Payment_Method":invoice.get("Payment_Method"),       
                "IBAN":invoice.get("IBAN"),
                "Invoice_Description": invoice.get("description"),
                "OCR_Confidence": invoice.get("ocr_understanding_confidence"),
                "Error": error,
            }
            rows.append(row)
    print(rows)
    # Create a DataFrame
    df = pd.DataFrame(rows)


    with sqlite3.connect("supplier_invoices.db") as conn:
        df.to_sql("invoices", conn, if_exists="append", index=False)
    

    conn.close()



## __Main__ Test

In [8]:
pdf_path = "./test1.pdf"

with open(pdf_path, "rb") as f:
    pdf_bytes = f.read()

invoiceDate =extract_invoice_data_from_pdf(pdf_bytes)



write_invoices_to_db_structured(invoiceDate)

⏱️ OpenAI API call time: 16.20 seconds


In [52]:
import os

directory_path = "./test"
results = []

for filename in os.listdir(directory_path):
    # Skip hidden files and non-PDFs
    
    if filename.startswith('.') or not filename.lower().endswith(".pdf"):
        continue

    pdf_path = os.path.join(directory_path, filename)
    
    try:
        with open(pdf_path, "rb") as f:
            pdf_bytes = f.read()

        invoice_data = extract_invoice_data_from_pdf(pdf_bytes)
        invoice_data["source_file"] = filename
        # No error, so pass empty string
        write_invoices_to_db_structured(invoice_data, error="")

    except Exception as e:
        # In case of error, store only the filename and the error message
        invoice_data = {"source_file": filename}
        print(f"Error processing {filename}: {e}")  # Optional: print to console
        write_invoices_to_db_structured(invoice_data, error=str(e))



# Now `results` contains both successes and failures


fileName .DS_Store
fileName 20220705131924.pdf
⏱️ OpenAI API call time: 13.86 seconds
invoice_data_file_name 20220705131924.pdf
{'invoice_number': '#NL-EU357528', 'invoice_date': 'Jun 13, 2022', 'provider': {'name': 'Ubiquiti Store Europe', 'address': 'eu.store.ui.com', 'contact': {'email': 'eu.store@ui.com', 'phone': None, 'fax': None, 'mobile': None}, 'VAT_NUMBER/NIE/CIF': 'ESB55606446'}, 'payment_method': 'Visa (1562)', 'items': [{'code': 'ES-8-150W-EU', 'quantity': 2, 'description': 'EdgeSwitch 8 150W', 'english_description': 'EdgeSwitch 8 150W', 'isServiceOrProduct': 'product', 'category': 'Networking Equipment', 'unit': None, 'unit_price': 189.0, 'total': 378.0}, {'code': 'UAP-AC-M-PRO-5', 'quantity': 1, 'description': 'Access Point AC Mesh Pro 5-Pack', 'english_description': 'Access Point AC Mesh Pro 5-Pack', 'isServiceOrProduct': 'product', 'category': 'Networking Equipment', 'unit': None, 'unit_price': 901.0, 'total': 901.0}], 'total': {'base_amount': 1288.87, 'VAT_rate': 0, '

## Main Directory

### processing_log Table

In [71]:
# SQLite DB setup
db_path = "supplier_invoices.db"  
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create processing log table if it doesn't exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS processing_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        directory TEXT,
        file TEXT,
        start_time TEXT,
        duration_sec REAL,
        status TEXT,
        error TEXT
    )
''')
conn.commit()

In [72]:
import os
import time
import traceback



# Base path where all directories exist
base_path = "./test_2022"  # Replace with your actual base folder path

# Get all subdirectories
all_directories = [
    os.path.join(base_path, d)
    for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d))
]

# Process each directory
for directory_path in all_directories:
    directory_name = os.path.basename(directory_path)

    # Get sorted list of PDF files
    pdf_files = sorted([
        f for f in os.listdir(directory_path)
        if f.lower().endswith(".pdf") and not f.startswith('.')
    ])

    for filename in pdf_files:
        pdf_path = os.path.join(directory_path, filename)
        start_time = time.strftime("%Y-%m-%d %H:%M:%S")
        t0 = time.time()
        error_msg = ""
        status = "success"

        source_file_name = f"{directory_name}_{filename}"

        try:
            with open(pdf_path, "rb") as f:
                pdf_bytes = f.read()

            # Assuming this function parses your invoice data
            invoice_data = extract_invoice_data_from_pdf(pdf_bytes)
            invoice_data["source_file"] = source_file_name

            # Save invoice data to DB
            write_invoices_to_db_structured(invoice_data, error="")

        except Exception as e:
            error_msg = traceback.format_exc()
            status = "failed"
            invoice_data = {"source_file": source_file_name}

            # Save error to invoice table too
            write_invoices_to_db_structured(invoice_data, error=str(e))

        finally:
            duration = round(time.time() - t0, 2)

            # Write log entry
            cursor.execute('''
                INSERT INTO processing_log (directory, file, start_time, duration_sec, status, error)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (
                directory_path,
                filename,
                start_time,
                duration,
                status,
                error_msg
            ))
            conn.commit()

# Cleanup
conn.close()


⏱️ OpenAI API call time: 12.88 seconds
[{'Source_File': 'ABRIL 2022_0518-22.pdf', 'Invoice_Number': '2022/1047', 'Invoice_Date': '2022-03-02', 'Invoice_Series': None, 'Invoice_Supply_Date': None, 'Provider_Name': 'MassivePixel - Unipessoal, Lda', 'Provider_Email': 'massivepixellda@gmail.com', 'Provider_Phone': None, 'Provider_Fax': None, 'Provider_Mobile': None, 'Provider_Address': 'Rua Patrão Sérgio nº62 3ºDto, 4490-579 Póvoa de Varzim, Portugal', 'Provider_VAT': '514164972', 'Item_Code': 'WD Gold', 'Item_Description': '3,84 TB PCIe Gen. 3 Enterprise SSD, Gran resistencia: 5600.TBW, Amarillo', 'Item_English_Description': '3.84 TB PCIe Gen. 3 Enterprise SSD, High endurance: 5600.TBW, Yellow', 'Item_Category': 'Computers & Accessories', 'Item_AiSuggested_Category': 'Enterprise SSD Storage', 'Item_Matched_Category': 'Computers & Accessories', 'Item_Generic_Category': 'Storage', 'Item_Type': 'product', 'Item_Quantity': 1, 'Item_Unit': None, 'Item_Unit_Price': 482.21, 'Discount_Rate': None